# ML Pipeline

Bu notebook tüm fraud detection sürecini production-ready bir pipeline haline getiriyor. Neden pipeline? Çünkü notebook'ta ayrı ayrı çalışan adımlar production'da senkronize olmazsa felaket. Training'de yaptığın bir preprocessing'i inference'da unutursan, model bambaşka data görüyor ve çöp sonuç üretiyor.

Feature Engineering'de türettiğimiz 36 feature ve Optuna ile optimize ettiğimiz LightGBM parametreleri tek bir sklearn Pipeline'ında birleşiyor. Böylece `pipeline.fit()` ile train edip `pipeline.predict()` ile production'da kullanıyoruz - arada hiçbir adım kaybolmuyor.

In [1]:
import pandas as pd
import numpy as np
import warnings
import os
import json
import joblib

from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score, GroupKFold
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)
print("Setup complete")

Setup complete


## 1. Veri ve Parametreler

In [ ]:
data_dir = '../../data/processed'
model_dir = '../../models/fraud_detection'

X_train_full = pd.read_csv(f'{data_dir}/X_train.csv')
X_val_full = pd.read_csv(f'{data_dir}/X_val.csv')
y_train = pd.read_csv(f'{data_dir}/y_train.csv').squeeze()
y_val = pd.read_csv(f'{data_dir}/y_val.csv').squeeze()

# ============================================================
# FEATURE SELECTION: SHAP + Native Rank Sum Threshold = 66
# 06_ModelEvaluation'da belirlenen optimum threshold
# rank_sum > 66 olan feature'lar çıkarılıyor
# AUC: 0.8842 | Recall: 0.6513 | Precision: 0.2512 | F1: 0.3625
# ============================================================
# Excluded feature'lar (rank_sum > 66):
excluded_features = ['addr2_freq', 'card3_freq']  # rank_sum: 69, 71
selected_features = [f for f in X_train_full.columns if f not in excluded_features]

X_train = X_train_full[selected_features]
X_val = X_val_full[selected_features]

print(f"Feature Selection (SHAP + Native Rank Sum <= 66):")
print(f"  Original: {X_train_full.shape[1]} features")
print(f"  Selected: {len(selected_features)} features")
print(f"  Excluded: {excluded_features}")

# Load TransactionDT for time-based validation (same as 05_ModelOptimization)
train_full = pd.read_csv('../../data/train_optimized.csv', usecols=['TransactionDT'])

# Create month_num (approximately 30 days per month)
train_full['month_num'] = (train_full['TransactionDT'] // (86400 * 30)).astype('int16')

# Align with X_train + X_val indices
month_train = train_full['month_num'].iloc[:len(X_train)].values
month_val = train_full['month_num'].iloc[len(X_train):len(X_train)+len(X_val)].values

with open(f'{model_dir}/optimized/optimization_metadata.json', 'r') as f:
    optimization_meta = json.load(f)
lgb_params = optimization_meta['lightgbm']['best_params']

print(f"\nTrain: {X_train.shape} | Val: {X_val.shape}")
print(f"Fraud rate: {y_train.mean()*100:.2f}%")
print(f"Month range: {month_train.min()} to {month_val.max()}")

Feature Selection (SHAP + Native Rank Sum <= 66):
  Original: 36 features
  Selected: 35 features
  Excluded: ['addr2_freq']

Train: (354324, 35) | Val: (118108, 35)
Fraud rate: 3.38%
Month range: 0 to 4

Train: (354324, 35) | Val: (118108, 35)
Fraud rate: 3.38%
Month range: 0 to 4


## 2. Pipeline Yapısı ve Model Seçimi

**Neden LightGBM?** 05_ModelOptimization'da CatBoost (AUC: 0.8717) ve LightGBM'i (AUC: 0.8800) 100 Optuna trial ile karşılaştırdık. LightGBM baseline'a (0.8766) göre +0.0034 iyileşme sağlarken, CatBoost -0.0049 geriledi. PR-AUC ve MCC metriklerinde de LightGBM önde.

**Neden 35 Feature?** 06_ModelEvaluation'da SHAP ve Native importance rank'larının toplamını analiz ettik. Threshold=66 ile `addr2_freq` (rank_sum=69) çıkarıldı - bu feature her iki metrikte de düşük sırada, modele anlamlı katkı sağlamıyor. 35 feature ile AUC: 0.8842 elde ettik.

In [3]:
# Train model with new feature set (using Optuna params)
# lgb_params'dan class_weight varsa çıkar, biz kendi değerimizi kullanacağız
params_clean = {k: v for k, v in lgb_params.items() if k not in ['class_weight', 'random_state', 'verbose', 'n_jobs']}

model = LGBMClassifier(
    **params_clean,
    class_weight='balanced',
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Create pipeline
pipeline = Pipeline([
    ('classifier', model)
])

print("Pipeline structure:")
for name, step in pipeline.named_steps.items():
    print(f"  {name}: {type(step).__name__}")
print(f"\nTrained on {X_train.shape[1]} features (threshold=66 selection)")
print(f"Model params: n_estimators={model.n_estimators}, max_depth={model.max_depth}")

Pipeline structure:
  classifier: LGBMClassifier

Trained on 35 features (threshold=66 selection)
Model params: n_estimators=751, max_depth=8


## 3. Eğitim ve Değerlendirme

In [4]:
# Model already trained in 05'te train edildi, sadece tahmin yap
y_train_proba = pipeline.predict_proba(X_train)[:, 1]
y_val_proba = pipeline.predict_proba(X_val)[:, 1]
y_val_pred = pipeline.predict(X_val)

train_auc = roc_auc_score(y_train, y_train_proba)
val_auc = roc_auc_score(y_val, y_val_proba)

print(f"Using pre-trained optimized model from 05_ModelOptimization")
print(f"\nTrain AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f} | Gap: {train_auc - val_auc:.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred):.4f} | Recall: {recall_score(y_val, y_val_pred):.4f} | F1: {f1_score(y_val, y_val_pred):.4f}")
print(f"\nExpected Val AUC: 0.8800 (from 05_ModelOptimization metadata)")

Using pre-trained optimized model from 05_ModelOptimization

Train AUC: 0.9891 | Val AUC: 0.8786 | Gap: 0.1105
Precision: 0.2498 | Recall: 0.6311 | F1: 0.3579

Expected Val AUC: 0.8800 (from 05_ModelOptimization metadata)


## 4. Cross-Validation (Time-Based)

05_ModelOptimization ile aynı stratejiyi kullanıyoruz: GroupKFold with months. Neden? Fraud pattern'ları zamanla değişiyor ve gerçek dünyada model her zaman geçmiş veriyle eğitilip gelecek işlemleri tahmin edecek. Random split yapsak data leakage olur - Kasım verisinden öğrenip Ekim'i tahmin etmek gerçekçi değil. Time-based CV bu sorunu çözüyor.

In [5]:
# Combine train+val for CV (same as 05_ModelOptimization)
X_full = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_full = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)
month_full = np.concatenate([month_train, month_val])

# Time-based GroupKFold - same strategy as 05_ModelOptimization
cv = GroupKFold(n_splits=3)

print(f"CV Strategy: GroupKFold with {cv.n_splits} splits (month-based)")
print(f"Groups: month_num (prevents future data leakage)")

# Verify fold distribution
for i, (train_idx, val_idx) in enumerate(cv.split(X_full, y_full, groups=month_full)):
    train_months = np.unique(month_full[train_idx])
    val_months = np.unique(month_full[val_idx])
    print(f"  Fold {i+1}: Train months {train_months.tolist()} -> Val months {val_months.tolist()}")

# Manual CV (GroupKFold doesn't work with cross_val_score directly for groups)
cv_scores = []
for train_idx, val_idx in cv.split(X_full, y_full, groups=month_full):
    X_tr, X_vl = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_tr, y_vl = y_full.iloc[train_idx], y_full.iloc[val_idx]
    
    temp_model = LGBMClassifier(**{**lgb_params, 'class_weight': 'balanced', 'random_state': 42, 'verbose': -1, 'n_jobs': -1})
    temp_model.fit(X_tr, y_tr)
    cv_scores.append(roc_auc_score(y_vl, temp_model.predict_proba(X_vl)[:, 1]))

cv_scores = np.array(cv_scores)
print(f"\nCV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"Folds: {[f'{s:.4f}' for s in cv_scores]}")

CV Strategy: GroupKFold with 3 splits (month-based)
Groups: month_num (prevents future data leakage)
  Fold 1: Train months [1, 2, 3, 4] -> Val months [0]
  Fold 2: Train months [0, 1, 2] -> Val months [3, 4]
  Fold 3: Train months [0, 3, 4] -> Val months [1, 2]

CV AUC: 0.8598 (+/- 0.0152)
Folds: ['0.8389', '0.8662', '0.8743']

CV AUC: 0.8598 (+/- 0.0152)
Folds: ['0.8389', '0.8662', '0.8743']


In [ ]:
# ============================================================
# BASELINE vs PIPELINE COMPARISON
# 03_Modeling (class_weight) vs 07_Pipeline (Optimized)
# ============================================================
print("=" * 80)
print("BASELINE (03_Modeling) vs PIPELINE (07_MLPipeline)")
print("=" * 80)

# 03_Modeling Results (class_weight version)
baseline_n_features_full = 392      # 03_Modeling full model - all feature'lar
baseline_n_features_interp = 16     # 03_Modeling interpretable model

baseline_full_auc = 0.9187
baseline_full_precision = 0.2814
baseline_full_recall = 0.7237
baseline_full_f1 = 0.4052

baseline_interp_auc = 0.8453
baseline_interp_precision = 0.1538
baseline_interp_recall = 0.6647
baseline_interp_f1 = 0.2498

# Pipeline metrics
pipeline_n_features = X_train.shape[1]
precision_val = precision_score(y_val, y_val_pred)
recall_val = recall_score(y_val, y_val_pred)
f1_val = f1_score(y_val, y_val_pred)

print(f"\n03_Modeling:")
print(f"  - Full Model: {baseline_n_features_full} feature (tüm original: V, id_, D, M, C dahil)")
print(f"  - Interpretable Model: {baseline_n_features_interp} feature (sadece yorumlanabilir)")
print(f"\n07_Pipeline (04_FE + 05_Optuna + 06_SHAP selection):")
print(f"  - Pipeline: {pipeline_n_features} feature (threshold=66 ile seçilmiş)")

print(f"\n{'Model':<40} {'Features':>8} {'AUC':>8} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 84)
print(f"{'03_Baseline Full (all features)':<40} {baseline_n_features_full:>8} {baseline_full_auc:>8.4f} {baseline_full_precision:>10.4f} {baseline_full_recall:>8.4f} {baseline_full_f1:>8.4f}")
print(f"{'03_Baseline Interp (yorumlanabilir)':<40} {baseline_n_features_interp:>8} {baseline_interp_auc:>8.4f} {baseline_interp_precision:>10.4f} {baseline_interp_recall:>8.4f} {baseline_interp_f1:>8.4f}")
print("-" * 84)
print(f"{'07_Pipeline (SHAP+Native threshold=66)':<40} {pipeline_n_features:>8} {val_auc:>8.4f} {precision_val:>10.4f} {recall_val:>8.4f} {f1_val:>8.4f}")

print(f"\n{'='*80}")
print("İYİLEŞTİRME ANALİZİ")
print(f"{'='*80}")

# Percentage calculations
auc_pct = (val_auc - baseline_interp_auc) / baseline_interp_auc * 100
recall_pct = (recall_val - baseline_interp_recall) / baseline_interp_recall * 100
f1_pct = (f1_val - baseline_interp_f1) / baseline_interp_f1 * 100

print(f"\n03_Interp -> 07_Pipeline:")
print(f"  Features: {baseline_n_features_interp} -> {pipeline_n_features} ({pipeline_n_features - baseline_n_features_interp:+d} feature engineering ile)")
print(f"  AUC:      {baseline_interp_auc:.4f} -> {val_auc:.4f} ({val_auc - baseline_interp_auc:+.4f}, %{auc_pct:+.1f})")
print(f"  Recall:   {baseline_interp_recall:.4f} -> {recall_val:.4f} ({recall_val - baseline_interp_recall:+.4f}, %{recall_pct:+.1f})")
print(f"  F1:       {baseline_interp_f1:.4f} -> {f1_val:.4f} ({f1_val - baseline_interp_f1:+.4f}, %{f1_pct:+.1f})")

# Full vs Pipeline comparison
auc_full_pct = (val_auc - baseline_full_auc) / baseline_full_auc * 100

print(f"\n03_Full vs 07_Pipeline:")
print(f"  Features: {baseline_n_features_full} -> {pipeline_n_features} ({pipeline_n_features - baseline_n_features_full:+d})")
print(f"  AUC:      {baseline_full_auc:.4f} -> {val_auc:.4f} ({val_auc - baseline_full_auc:+.4f}, %{auc_full_pct:+.1f})")

print(f"\nSonuç: Interpretable {baseline_n_features_interp} değişkenden {pipeline_n_features} feature türettik")
print(f"       ve black-box V features olmadan bile Full baseline'a yakın performans elde ettik!")

BASELINE (03_Modeling) vs PIPELINE (07_MLPipeline)


NameError: name 'n_selected' is not defined

## 6. Pipeline Kaydetme

Tüm model ve preprocessing adımları tek pkl dosyasında. Production'da `joblib.load()` ile yükle, `predict_proba()` ile tahmin al - bu kadar basit. Feature sırası, model parametreleri, her şey kapsüllenmiş durumda.

In [ ]:
pipeline_dir = '../../models/fraud_detection/pipeline'
os.makedirs(pipeline_dir, exist_ok=True)

joblib.dump(pipeline, f'{pipeline_dir}/lgb_pipeline.pkl')

# Metadata
metadata = {
    'lgb_pipeline': {
        'val_auc': float(val_auc),
        'precision': float(precision_val),
        'recall': float(recall_val),
        'f1': float(f1_val),
        'n_features': int(X_train.shape[1]),
        'feature_columns': X_train.columns.tolist(),
        'excluded_features': excluded_features,
        'feature_selection': 'SHAP + Native Rank Sum <= 66',
        'source': '05_ModelOptimization params + 06_ModelEvaluation feature selection'
    },
    'model_selection_reason': 'LightGBM selected: AUC 0.8800 vs CatBoost 0.8717 in 05_ModelOptimization'
}

with open(f'{pipeline_dir}/pipeline_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved to:", pipeline_dir)
for f in os.listdir(pipeline_dir):
    print(f"  {f} ({os.path.getsize(f'{pipeline_dir}/{f}')/1024:.1f} KB)")

## 7. Yükleme Testi

In [ ]:
loaded = joblib.load(f'{pipeline_dir}/lgb_pipeline.pkl')
loaded_auc = roc_auc_score(y_val, loaded.predict_proba(X_val)[:, 1])

print(f"Original: {val_auc:.4f} | Loaded: {loaded_auc:.4f} | Match: {'Yes' if abs(val_auc - loaded_auc) < 0.0001 else 'No'}")

sample_pred = loaded.predict_proba(X_val.iloc[[0]])[0, 1]
print(f"Sample prediction: {sample_pred:.4f} (actual: {y_val.iloc[0]})")

## 8. Production Inference

In [ ]:
def predict_fraud(data, pipeline_path=None, threshold=0.5):
    if pipeline_path is None:
        pipeline_path = '../../models/fraud_detection/pipeline/lgb_pipeline.pkl'
    
    pipe = joblib.load(pipeline_path)
    proba = pipe.predict_proba(data)[:, 1]
    pred = (proba >= threshold).astype(int)
    risk = pd.cut(proba, bins=[0, 0.1, 0.3, 0.5, 0.7, 1.0],
                  labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
    
    return {'predictions': pred, 'probabilities': proba, 'risk_categories': risk}

# Test
results = predict_fraud(X_val.head(15))
print(pd.DataFrame({
    'Prob': results['probabilities'].round(3),
    'Pred': results['predictions'],
    'Risk': results['risk_categories'],
    'Actual': y_val.head(15).values
}).to_string(index=False))

## Sonuç ve Model Seçimi Gerekçesi

**Neden LightGBM?** 05_ModelOptimization'da 100 Optuna trial ile karşılaştırma yaptık:
- LightGBM: Val AUC = 0.8800, PR-AUC = 0.374, MCC = 0.363
- CatBoost: Val AUC = 0.8717, PR-AUC = 0.302, MCC = 0.315
- Baseline: Val AUC = 0.8766

LightGBM tüm metriklerde önde. Ayrıca eğitim süresi CatBoost'tan kısa.

**Neden bu feature set?** 
- Orijinal 392 feature'dan 16 interpretable feature seçtik
- Bu 16 feature'dan 36 yeni feature türettik
- 06_ModelEvaluation'daki SHAP + Native Importance analizi ile feature rank sum hesaplandı
- **Threshold=66**: Rank sum ≤ 66 olan feature'lar seçildi → **35 feature** (sadece `addr2_freq` elendi)
- Bu threshold'da: AUC=0.8842, Recall=0.6513, Precision=0.2512, F1=0.3625

**Production dosyaları:**
- `lgb_pipeline.pkl`: 35 feature, Optuna optimize + SHAP feature selection
- `pipeline_metadata.json`: Feature listesi, excluded features ve performans metrikleri

Model `models/fraud_detection/pipeline/` dizininde deployment'a hazır.